In [1]:
import json
import os
import random
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# 1. 환경 설정
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

INPUT_FILE = "kmi.json"
OUTPUT_FILE = "integrated_benchmark.json"
TARGET_COUNT = 50  # 생성할 페르소나 개수 (총 문항 수는 이보다 많음)

# 2. 벤치마크 생성 프롬프트 (GPT-4o-mini)
GEN_PROMPT = """
당신은 임상 심리 벤치마크 출제자입니다.
제공된 [상담 데이터]를 분석하여, AI 상담 내담자 모델을 평가하기 위한 '테스트 케이스 세트'를 생성하세요.

[입력 데이터]
- 카테고리: {category}
- 대화 요약: {dialogue_text}

[생성 요청]
다음 JSON 형식으로 4개의 테스트 케이스를 생성하세요:

1. **persona**: 내담자의 나이, 직업, 핵심 증상, 성격을 1문장으로 요약.
2. **cases**: [
    {{
        "group": "Standard", "metric": "Prompt-to-Line", "phase": "All",
        "query": "(페르소나 확인 질문, 예: 자기소개)", "criteria": "(이름/직업 등 설정 유지)"
    }},
    {{
        "group": "K-Patient", "metric": "Resistance", "phase": "Early",
        "query": "(상담 초기 탐색 질문)", "criteria": "방어적, 단답형, 외부 탓, 불신 (저항)"
    }},
    {{
        "group": "K-Patient", "metric": "Change Talk", "phase": "Late",
        "query": "(변화나 통찰 유도 질문)", "criteria": "변화 의지(Desire/Ability), 통찰, 구체적 계획 (수용)"
    }}
]

**반드시 JSON 포맷만 출력하세요.**
"""

def generate_bench_case(category, dialogue):
    # 대화 내용을 텍스트로 변환 (앞부분만 참조)
    dialogue_text = "\n".join([f"{t['role']}: {t.get('utterance_ko','')}" for t in dialogue[:8]])
    
    try:
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": GEN_PROMPT.format(category=category, dialogue_text=dialogue_text)}],
            response_format={"type": "json_object"}
        )
        return json.loads(res.choices[0].message.content)
    except:
        return None

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ {INPUT_FILE} 파일이 없습니다.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    # 다양한 카테고리가 섞이도록 셔플
    random.shuffle(raw_data)
    
    final_benchmark = []
    
    print(f"🚀 벤치마크 데이터 확장 시작 (목표 페르소나: {TARGET_COUNT}개)...")
    
    count = 0
    for sample in tqdm(raw_data):
        if count >= TARGET_COUNT: break
            
        dlg = sample.get('dialogue', [])
        cat = sample.get('category_ko', '일반')
        
        if len(dlg) < 4: continue
        
        # GPT-4o-mini로 케이스 생성
        data = generate_bench_case(cat, dlg)
        
        if data and 'persona' in data and 'cases' in data:
            persona = data['persona']
            for case in data['cases']:
                # 공통 필드 추가
                case['persona'] = persona
                final_benchmark.append(case)
            count += 1

    # 저장
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(final_benchmark, f, indent=2, ensure_ascii=False)
        
    print(f"✅ 벤치마크 생성 완료! 총 {len(final_benchmark)}개 문항이 '{OUTPUT_FILE}'에 저장되었습니다.")

if __name__ == "__main__":
    main()

🚀 벤치마크 데이터 확장 시작 (목표 페르소나: 50개)...


 14%|█▎        | 135/1000 [33:38<3:35:30, 14.95s/it]

✅ 벤치마크 생성 완료! 총 150개 문항이 'integrated_benchmark.json'에 저장되었습니다.
